In [5]:
#Linear Regressionfrom sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
d = load_diabetes(); Xr, yr = d.data, d.target
Xtr, Xte, ytr, yte = train_test_split(Xr, yr, test_size=0.2, random_state=0)

w1 = fit_normal_equation(Xtr, ytr)
w2, h2 = fit_gradient_descent(Xtr, ytr, lr=0.1, epochs=5000)
skr = LinearRegression().fit(Xtr, ytr)
w3 = np.concatenate([[skr.intercept_], skr.coef_])
print(np.allclose(w1, w3), np.abs(w2 - w3).max())
print(r2(yte, predict(Xte, w1)), skr.score(Xte, yte))
import numpy as np

from sklearn.linear_model import LinearRegression

def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])

def fit_normal_equation(X, y):
    Xd = add_intercept(X)
    return np.linalg.solve(Xd.T @ Xd, Xd.T @ y)

def fit_gradient_descent(X, y, lr=0.1, epochs=1000):
    mu, sd = X.mean(axis=0), X.std(axis=0)        # 1. remember the scaling
    Z = (X - mu) / sd                             # 2. standardise each column
    Zd = add_intercept(Z)
    n = len(y); w = np.zeros(Zd.shape[1]); history = []
    for _ in range(epochs):                       # 3. the gradient descent loop
        error = Zd @ w - y
        history.append((error ** 2).mean())       #    record the MSE
        grad = (2 / n) * Zd.T @ error             #    gradient of MSE (Topic 4)
        w = w - lr * grad                         #    step downhill
    slopes = w[1:] / sd                           # 4. convert back to original units
    intercept = w[0] - (slopes * mu).sum()
    return np.concatenate([[intercept], slopes]), history

def predict(X, w):
    return add_intercept(X) @ w

def r2(y, yhat):
    return 1 - ((y - yhat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

In [6]:
rng = np.random.default_rng(42); n = 200
size = rng.uniform(40, 150, n); rooms = rng.integers(1, 6, n)
price = 50 + 2.0*size + 15*rooms + rng.normal(0, 20, n)
X = np.column_stack([size, rooms])

w_ne = fit_normal_equation(X, price)
w_gd, hist = fit_gradient_descent(X, price)
sk = LinearRegression().fit(X, price)
w_sk = np.concatenate([[sk.intercept_], sk.coef_])
print(w_ne, w_gd, w_sk)
print(np.allclose(w_ne, w_sk), np.allclose(w_gd, w_sk))

[52.75909908  1.96936819 15.24713236] [52.75909908  1.96936819 15.24713236] [52.75909908  1.96936819 15.24713236]
True True


In [8]:
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def pca_fit(X, k):
    mu, sd = X.mean(axis=0), X.std(axis=0)
    Z = (X - mu) / sd                          # 1. standardise
    C = np.cov(Z, rowvar=False)                # 2. covariance matrix (13×13)
    vals, vecs = np.linalg.eigh(C)             # 3. eigen-decomposition
    order = np.argsort(vals)[::-1]             # 4. sort biggest → smallest
    vals, vecs = vals[order], vecs[:, order]
    return {"mu": mu, "sd": sd, "components": vecs[:, :k],
            "eigenvalues": vals, "explained_ratio": vals / vals.sum()}

def pca_transform(X, model):
    Z = (X - model["mu"]) / model["sd"]        # use the SAME mu, sd as fitting
    return Z @ model["components"]             # 5. project onto the top-k directions

w = load_wine(); X, y = w.data, w.target
m = pca_fit(X, 2)
scores = pca_transform(X, m)
print(m["explained_ratio"].round(4), np.cumsum(m["explained_ratio"]).round(4))

[0.362  0.1921 0.1112 0.0707 0.0656 0.0494 0.0424 0.0268 0.0222 0.0193
 0.0174 0.013  0.008 ] [0.362  0.5541 0.6653 0.736  0.8016 0.851  0.8934 0.9202 0.9424 0.9617
 0.9791 0.992  1.    ]


In [ ]:
Zs = StandardScaler().fit_transform(X)
sk = PCA(n_components=2).fit(Zs)
print(sk.explained_variance_ratio_)                                    # [0.362  0.1921] ✅
print(np.allclose(np.abs(sk.components_.T), np.abs(m["components"])))  # True
print(scores[:3]); print(sk.transform(Zs)[:3])

In [9]:
Z = (X - X.mean(0)) / X.std(0)
U, s, Vt = np.linalg.svd(Z, full_matrices=False)
print((s**2 / (len(X) - 1))[:4])                 # [4.732 2.511 1.454 0.924] = your eigenvalues ✅
print(np.corrcoef(scores.T)[0, 1])               # 0.0  → PCs are uncorrelated
print(scores.var(axis=0, ddof=1))                # [4.732 2.511] = eigenvalues
Zhat = scores @ m["components"].T                # rebuild 13 columns from 2 PCs
print(((Z - Zhat)**2).mean())                    # 0.4459

[4.73243698 2.51108093 1.45424187 0.92416587]
-2.8002034776402998e-18
[4.73243698 2.51108093]
0.4459366164306474
